# Ampliación del inventario: R002, R004 y R007

Este notebook comprueba los trazados e indicadores de elevación de las primeras rutas de ampliación. R004 y R007 tienen calidad preliminar apta. R002 conserva calidad dudosa porque la geometría es compatible con Los Pantzas, pero las vías OSM no incluyen ese nombre.

In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

proyecto = Path.cwd().resolve()
if not (proyecto / '01_documentacion').exists():
    proyecto = proyecto.parent
assert (proyecto / '01_documentacion').exists()
proyecto

WindowsPath('C:/Users/LENOVO/Documents/ChatGPT/tesis_trekking_pichincha')

## Estado documental

In [2]:
inventario = pd.read_csv(proyecto / '01_documentacion' / 'inventario_rutas.csv')
ids_ampliacion = ['R002', 'R004', 'R007']
ampliacion = inventario[inventario['id_ruta'].isin(ids_ampliacion)].copy()
columnas = [
    'id_ruta', 'nombre_ruta', 'canton', 'tipo_recorrido',
    'trazado_disponible', 'elevacion_disponible',
    'calidad_preliminar', 'decision'
]
assert len(ampliacion) == 3
assert ampliacion['id_ruta'].is_unique
assert ampliacion['trazado_disponible'].eq('si').all()
assert ampliacion['elevacion_disponible'].eq('si').all()
calidad = ampliacion.set_index('id_ruta')['calidad_preliminar']
assert calidad.loc['R002'] == 'dudosa'
assert calidad.loc[['R004', 'R007']].eq('apta').all()
ampliacion[columnas]

,id_ruta,nombre_ruta,canton,tipo_recorrido,trazado_disponible,elevacion_disponible,calidad_preliminar,decision
1,R002,Sendero Los Pantzas,Mejía,ida_y_vuelta,si,si,dudosa,pendiente
3,R004,Circuito Pondoña (Sendero Pondoña),Quito,circular,si,si,apta,pendiente
6,R007,Sendero sur al volcán El Corazón,Mejía,ida_y_vuelta,si,si,apta,pendiente


## Validación geométrica

In [3]:
archivos_trazado = {
    'R002': ('R002_trazado_ida_continuo_2026-10-06.geojson', 32717),
    'R004': ('R004_trazado_circular_continuo_2026-10-06.geojson', 32617),
    'R007': ('R007_trazado_ida_continuo_2026-10-06.geojson', 32717),
}
filas = []
for id_ruta, (nombre_archivo, epsg_metrico) in archivos_trazado.items():
    trazado = gpd.read_file(
        proyecto / '03_datos_intermedios' / 'trazados' / nombre_archivo
    )
    geometria = trazado.geometry.iloc[0]
    assert len(trazado) == 1
    assert trazado.crs.to_epsg() == 4326
    assert geometria.geom_type == 'LineString'
    assert geometria.is_valid
    assert geometria.is_simple
    if id_ruta == 'R004':
        assert geometria.is_closed
    filas.append({
        'id_ruta': id_ruta,
        'epsg_metrico': epsg_metrico,
        'distancia_trazado_km': trazado.to_crs(epsg_metrico).length.iloc[0] / 1000,
        'geometria': geometria.geom_type,
        'valida': geometria.is_valid,
        'simple': geometria.is_simple,
        'cerrada': geometria.is_closed,
    })
geometrias = pd.DataFrame(filas).round(3)
geometrias

C:\Users\LENOVO\miniforge3\envs\trekking_pichincha\Lib\site-packages\pyogrio\core.py:38: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


,id_ruta,epsg_metrico,distancia_trazado_km,geometria,valida,simple,cerrada
0,R002,32717,6.060,LineString,True,True,False
1,R004,32617,1.663,LineString,True,True,True
2,R007,32717,3.838,LineString,True,True,False


## Indicadores de elevación

In [4]:
indicadores = pd.read_csv(
    proyecto / '07_resultados' / 'tablas' / 'indicadores_elevacion_ampliacion.csv'
)
indicadores = indicadores[indicadores['id_ruta'].isin(ids_ampliacion)].copy()
columnas_indicadores = [
    'id_ruta', 'tipo_recorrido', 'distancia_total_km',
    'puntos_perfil', 'valores_faltantes', 'elevacion_min_m',
    'elevacion_max_m', 'ascenso_total_m',
    'pendiente_media_abs_pct', 'pendiente_p95_abs_pct'
]
assert len(indicadores) == 3
assert indicadores['id_ruta'].is_unique
assert indicadores['valores_faltantes'].eq(0).all()
assert indicadores['elevacion_max_m'].gt(indicadores['elevacion_min_m']).all()
assert indicadores.set_index('id_ruta').loc['R004', 'tipo_recorrido'] == 'circular'
indicadores[columnas_indicadores]

,id_ruta,tipo_recorrido,distancia_total_km,puntos_perfil,valores_faltantes,elevacion_min_m,elevacion_max_m,ascenso_total_m,pendiente_media_abs_pct,pendiente_p95_abs_pct
0,R002,ida_y_vuelta,12.119,203,0,2809.673,3823.206,1040.427,17.170,47.383
1,R004,circular,1.663,57,0,2555.783,2745.402,192.821,23.187,72.919
2,R007,ida_y_vuelta,7.677,129,0,4026.883,4755.298,731.015,19.045,56.761


## Sensibilidad del ascenso acumulado

In [5]:
sensibilidad = pd.read_csv(
    proyecto / '07_resultados' / 'tablas' / 'sensibilidad_elevacion_ampliacion.csv'
)
sensibilidad = sensibilidad[sensibilidad['id_ruta'].isin(ids_ampliacion)]
comparacion = sensibilidad.pivot(
    index='id_ruta', columns='metodo', values='ascenso_total_m'
)
assert comparacion.shape == (3, 3)
assert comparacion.notna().all().all()
comparacion

metodo,mediana_3,mediana_5,raw
id_ruta,,,
R002,1040.427,1017.643,1189.507
R004,192.821,186.247,227.457
R007,731.015,728.415,749.415


## Comprobación de perfiles gráficos

In [6]:
perfiles = [
    proyecto / '07_resultados' / 'perfiles' / f'{id_ruta}_perfil_elevacion_2026-10-06.svg'
    for id_ruta in ids_ampliacion
]
assert all(archivo.exists() for archivo in perfiles)
[archivo.name for archivo in perfiles]

['R002_perfil_elevacion_2026-10-06.svg',
 'R004_perfil_elevacion_2026-10-06.svg',
 'R007_perfil_elevacion_2026-10-06.svg']

## Conclusión del primer lote

R004 cuenta con geometría y elevación para el análisis físico de trekking. R002 posee geometría y elevación completas, pero no se considerará una unidad definitiva hasta confirmar que las vías OSM corresponden exactamente al Sendero Los Pantzas. R007 también tiene perfil completo, aunque se excluye del modelado de trekking por corresponder a alta montaña.

# Consolidación del inventario (actualizada el 17 de octubre de 2026)

Las celdas anteriores documentan el primer lote R002/R004/R007. Esta sección reúne **todas las rutas R001–R026** procesadas hasta este corte, sin repetir la extracción del DEM. R020 y R021 son variantes de Yellow House Trail con tramos compartidos con R017; R022 se incorporó desde una instantánea OSM normalizada R023 desde XML OSM íntegro y R024/R025 (dos senderos al Mirador Cerro de la Luz) y R026 (Tarabita – Cascada Guarumos) desde XML OSM íntegro. Todas aún requieren comprobar el acceso. El inventario documental está en `01_documentacion/inventario_rutas.csv`; los indicadores y el estado de modelado proceden de `07_resultados/tablas/dataset_clasificacion_rutas.csv`, generado por `06_scripts/preparar_dataset_clasificacion.py`. Tener perfil de elevación **no equivale** a tener una clase de esfuerzo validada.

In [7]:
dataset = pd.read_csv(
    proyecto / '07_resultados' / 'tablas' / 'dataset_clasificacion_rutas.csv'
)
assert dataset['id_ruta'].is_unique
assert set(dataset['id_ruta']) == set(inventario['id_ruta'])
assert set(dataset['id_ruta']) == {f'R{i:03d}' for i in range(1, 28)}
columnas_resumen = [
    'id_ruta', 'nombre_ruta', 'nombre_destino', 'distancia_total_km',
    'ascenso_total_m', 'puntos_perfil', 'valores_faltantes', 'estado_modelado'
]
resumen = dataset.sort_values('id_ruta')[columnas_resumen].copy()
print('Rutas inventariadas:', len(resumen))
display(resumen.round(2))

Rutas inventariadas: 27


,id_ruta,nombre_ruta,nombre_destino,distancia_total_km,ascenso_total_m,puntos_perfil,valores_faltantes,estado_modelado
0,R001,Sendero Cumbre Rucu Pichincha,Rucu Pichincha,9.61,824.84,162,0,elegible_actual
1,R002,Sendero Los Pantzas,Refugio de Vida Silvestre Pasochoa,12.12,1040.43,203,0,no_elegible_aun
2,R003,Sendero Cascada Cóndor Machay,Cascada Cóndor Machay,7.48,448.98,126,0,elegible_actual
3,R004,Circuito Pondoña (Sendero Pondoña),Cerro Pondoña,1.66,192.82,57,0,elegible_actual
4,R005,Sendero Tarabita - Cascada Nambillo,Cascada Nambillo,1.90,165.42,33,0,no_elegible_aun
5,R006,Sendero al Ilaló,Cerro Ilaló,5.23,817.83,89,0,elegible_actual
6,R007,Sendero sur al volcán El Corazón,Volcán El Corazón,7.68,731.02,129,0,no_elegible_aun
7,R008,Sendero de ascenso al Pondoña,Cerro Pondoña,1.57,138.99,28,0,elegible_actual
8,R009,Sendero Principal de Maquipucuna,Reserva Maquipucuna,8.34,500.52,141,0,no_elegible_aun
9,R010,Sendero Tranquilo de Maquipucuna,Reserva Maquipucuna,3.19,313.05,55,0,no_elegible_aun


## Elegibilidad y pendientes de revisión

`elegible_actual` es una decisión documental para **entrar al análisis**, no una predicción de dificultad. R001 y R007 son de alta montaña; R002 necesita confirmar la identidad del sendero; R005, R009 y R010 tienen dudas sobre el trazado o la variante; R012 y R014 se excluyeron tras la revisión de Yanacocha. R011 y R013 figuran como elegibles provisionales y aún requieren confirmar su trazado. **Nota histórica:** en este corte todavía no se había entrenado la clasificación. El corte vigente alcanzó 20 rutas y su K-means se documenta y valida en `05_validacion_metodologica.ipynb`.

In [8]:
conteo = dataset['estado_modelado'].value_counts().rename_axis('estado_modelado').reset_index(name='rutas')
print('Resumen de elegibilidad:')
display(conteo)
print('Rutas que no entran todavía al modelado:')
display(dataset.loc[dataset['estado_modelado'] != 'elegible_actual',
                    ['id_ruta', 'nombre_ruta', 'calidad_preliminar', 'decision']]
        .sort_values('id_ruta'))

Resumen de elegibilidad:


,estado_modelado,rutas
0,elegible_actual,20
1,no_elegible_aun,7


Rutas que no entran todavía al modelado:


,id_ruta,nombre_ruta,calidad_preliminar,decision
1,R002,Sendero Los Pantzas,dudosa,pendiente
4,R005,Sendero Tarabita - Cascada Nambillo,dudosa,pendiente
6,R007,Sendero sur al volcán El Corazón,apta,pendiente
8,R009,Sendero Principal de Maquipucuna,dudosa,pendiente
9,R010,Sendero Tranquilo de Maquipucuna,dudosa,pendiente
11,R012,Sendero Andean Snipe,dudosa,excluir_provisional
13,R014,Trocha Inca,dudosa,excluir_provisional


## Dónde están los trazados y perfiles de cada ruta

Cada fila siguiente vincula el archivo GeoJSON de trabajo, el CSV de perfil procesado y el SVG de elevación. R001 conserva dos GeoJSON por la corrección histórica de su llegada a la cumbre. Los SVG son visualizaciones; el CSV procesado es el dato reutilizable para los indicadores. Esta comprobación es de existencia e integridad básica, no sustituye la validación de campo o de fuente.

In [9]:
carpeta_trazados = proyecto / '03_datos_intermedios' / 'trazados'
carpeta_procesados = proyecto / '04_datos_procesados' / 'perfiles'
carpeta_svg = proyecto / '07_resultados' / 'perfiles'
filas_archivos = []
for fila in dataset.sort_values('id_ruta').itertuples(index=False):
    trazados = sorted(carpeta_trazados.glob(f'{fila.id_ruta}_trazado_*.geojson'))
    svg = sorted(carpeta_svg.glob(f'{fila.id_ruta}_perfil_elevacion_*.svg'))
    csv_perfil = carpeta_procesados / fila.archivo_perfil
    filas_archivos.append({
        'id_ruta': fila.id_ruta,
        'geojson': ', '.join(a.name for a in trazados),
        'csv_perfil': csv_perfil.name if csv_perfil.exists() else 'FALTA',
        'svg_perfil': ', '.join(a.name for a in svg),
        'trazados_n': len(trazados), 'svg_n': len(svg),
        'csv_existe': csv_perfil.exists(),
    })
archivos_por_ruta = pd.DataFrame(filas_archivos)
assert archivos_por_ruta['trazados_n'].ge(1).all()
assert archivos_por_ruta['svg_n'].eq(1).all()
assert archivos_por_ruta['csv_existe'].all()
assert dataset['valores_faltantes'].eq(0).all()
display(archivos_por_ruta[['id_ruta', 'geojson', 'csv_perfil', 'svg_perfil']])
print('Comprobación básica superada para', len(archivos_por_ruta), 'rutas')

,id_ruta,geojson,csv_perfil,svg_perfil
0,R001,R001_trazado_subida_continuo_2026-10-01.geojso...,R001_perfil_elevacion_procesado_2026-10-05.csv,R001_perfil_elevacion_2026-10-05.svg
1,R002,R002_trazado_ida_continuo_2026-10-06.geojson,R002_perfil_elevacion_procesado_2026-10-06.csv,R002_perfil_elevacion_2026-10-06.svg
2,R003,R003_trazado_ida_continuo_2026-10-01.geojson,R003_perfil_elevacion_procesado_2026-10-05.csv,R003_perfil_elevacion_2026-10-05.svg
3,R004,R004_trazado_circular_continuo_2026-10-06.geojson,R004_perfil_elevacion_procesado_2026-10-06.csv,R004_perfil_elevacion_2026-10-06.svg
4,R005,R005_trazado_preliminar_2026-10-01.geojson,R005_perfil_elevacion_procesado_2026-10-05.csv,R005_perfil_elevacion_2026-10-05.svg
5,R006,R006_trazado_ida_continuo_2026-10-05.geojson,R006_perfil_elevacion_procesado_2026-10-05.csv,R006_perfil_elevacion_2026-10-05.svg
6,R007,R007_trazado_ida_continuo_2026-10-06.geojson,R007_perfil_elevacion_procesado_2026-10-06.csv,R007_perfil_elevacion_2026-10-06.svg
7,R008,R008_trazado_ida_continuo_2026-10-07.geojson,R008_perfil_elevacion_procesado_2026-10-07.csv,R008_perfil_elevacion_2026-10-07.svg
8,R009,R009_trazado_ida_continuo_2026-10-07.geojson,R009_perfil_elevacion_procesado_2026-10-07.csv,R009_perfil_elevacion_2026-10-07.svg
9,R010,R010_trazado_ida_continuo_2026-10-07.geojson,R010_perfil_elevacion_procesado_2026-10-07.csv,R010_perfil_elevacion_2026-10-07.svg


Comprobación básica superada para 27 rutas


## Ensayo exploratorio del 15 de octubre

Se muestran los resultados históricos del 17 de octubre calculados por `06_scripts/explorar_modelo_clasificacion.py`. Las 14 rutas de aquel corte eran provisionales; el ensayo no crea clases de dificultad ni recomendaciones. Se contrasta la muestra base con escenarios sin R011/R013, R020, R021 y R022 para comprobar sensibilidad a fuentes, pendientes y recorridos compartidos. El corte vigente de 15 rutas se registra al final del notebook.

In [10]:
evaluacion = pd.read_csv(
    proyecto / '07_resultados' / 'tablas'
    / 'evaluacion_clustering_exploratorio_sklearn_2026-10-17_r022.csv'
)
assert dataset['estado_modelado'].eq('elegible_actual').sum() >= 15
assert evaluacion.loc[evaluacion['escenario'].eq('base_4_variables'), 'n_rutas'].eq(14).all()
display(evaluacion[['escenario', 'n_rutas', 'k', 'silueta', 'tamano_minimo', 'ari_ward', 'ari_semillas_min', 'semillas_validas']])
print('Ensayo exploratorio; entrenamiento final bloqueado hasta revisar la muestra y sus fuentes.')

,escenario,n_rutas,k,silueta,tamano_minimo,ari_ward,ari_semillas_min,semillas_validas
0,base_4_variables,14,2,0.3607,3,0.6717,0.1391,30
1,base_4_variables,14,3,0.3968,2,0.5255,1.0000,30
2,base_4_variables,14,4,0.4197,2,0.7716,1.0000,30
3,base_5_variables,14,2,0.3253,3,0.1391,0.1391,30
4,base_5_variables,14,3,0.3514,3,1.0000,0.2918,30
5,base_5_variables,14,4,0.3356,2,0.4957,0.7872,22
6,sin_destino_d004,12,2,0.3815,3,0.6409,0.6409,30
7,sin_destino_d004,12,3,0.4056,2,0.5000,1.0000,30
8,sin_destino_d004,12,4,0.4975,2,0.6908,1.0000,30
9,sin_destino_d009,12,2,0.4033,3,1.0000,0.4005,30


Ensayo exploratorio; entrenamiento final bloqueado hasta revisar la muestra y sus fuentes.


## Puerta de cierre del 16 de octubre

El procesamiento de veinte trazados no equivale a veinte alternativas elegibles. Esta celda verifica el diagnóstico reproducible del hito; la decisión razonada está en `01_documentacion/decision_cierre_inventario_2026-10-16.md`. No se entrena ni publica un modelo final.

In [11]:
diagnostico = pd.read_csv(
    proyecto / '07_resultados' / 'tablas' / 'diagnostico_preparacion_modelo.csv'
)
muestra = diagnostico.set_index('control').loc['muestra_minima']
assert len(dataset) == 27
assert int(dataset['estado_modelado'].eq('elegible_actual').sum()) == 20
assert muestra['estado'] == 'OK'
assert int(muestra['valor']) == 20
display(diagnostico)
print('Meta cuantitativa alcanzada; validación metodológica y documental aún pendiente.')

,control,estado,valor,umbral,detalle
0,muestra_minima,OK,20.000,>=20,Número de alternativas elegibles para el ajust...
1,valores_faltantes,OK,0.000,0,Celdas faltantes en las cinco variables candid...
2,filas_duplicadas,OK,0.000,0,Perfiles físicos exactamente duplicados
3,variables_sin_variacion,OK,0.000,0,Variables que no separan ninguna ruta
4,correlacion_maxima_spearman,REVISAR,0.926,<0.85 preferible,Par más asociado: distancia_total_km y ascenso...


Meta cuantitativa alcanzada; validación metodológica y documental aún pendiente.


## EDA histórico del 17 de octubre: conjunto de 14 rutas

El informe `01_documentacion/eda_preliminar_14_rutas_2026-10-17.md` y la evidencia E33 contienen el corte histórico previo a R023. La nueva E34 documenta el conjunto actual de 15 rutas. Aquí se reproducen las cifras históricas sin entrenar ni publicar un modelo final.

In [12]:
variables_eda = ['distancia_total_km', 'ascenso_total_m', 'elevacion_max_m', 'pendiente_media_abs_pct', 'pendiente_p95_abs_pct']
elegibles_eda = dataset.loc[dataset['estado_modelado'].eq('elegible_actual') & ~dataset['id_ruta'].isin(['R001', 'R023', 'R024', 'R025', 'R026', 'R027'])].copy()
assert len(elegibles_eda) == 14 and len(dataset) >= 26
assert elegibles_eda[variables_eda].notna().all().all()
rho = elegibles_eda[variables_eda].corr(method='spearman').loc['distancia_total_km', 'ascenso_total_m']
assert 0.8 < rho < 1.0
alertas = []
for variable in variables_eda:
    q1, q3 = elegibles_eda[variable].quantile([0.25, 0.75])
    iqr = q3 - q1
    fuera = elegibles_eda.loc[(elegibles_eda[variable] < q1 - 1.5 * iqr) | (elegibles_eda[variable] > q3 + 1.5 * iqr), ['id_ruta', variable]]
    alertas.extend({'id_ruta': fila.id_ruta, 'variable': variable, 'valor': getattr(fila, variable)} for fila in fuera.itertuples(index=False))
assert all(fila['id_ruta'] in set(elegibles_eda['id_ruta']) for fila in alertas)
print(f'EDA histórico: {len(elegibles_eda)} rutas; Spearman distancia–ascenso={rho:.3f}; alertas={len(alertas)}. No hay clasificación final.')
display(pd.DataFrame(alertas))

EDA histórico: 14 rutas; Spearman distancia–ascenso=0.846; alertas=3. No hay clasificación final.


,id_ruta,variable,valor
0,R006,ascenso_total_m,817.832
1,R006,pendiente_media_abs_pct,31.262
2,R006,pendiente_p95_abs_pct,83.923


## Sensibilidad del agrupamiento a la composición del inventario

El ensayo del 17 de octubre retira por separado cada destino con al menos dos alternativas, además de R006. Son pruebas de fragilidad de la muestra, no clasificaciones válidas para recomendar rutas.

In [13]:
from sklearn.metrics import adjusted_rand_score
tabla_sensibilidad = pd.read_csv(proyecto / '07_resultados' / 'tablas' / 'evaluacion_clustering_exploratorio_sklearn_2026-10-17_r022.csv')
grupos_sensibilidad = pd.read_csv(proyecto / '07_resultados' / 'tablas' / 'grupos_exploratorios_sklearn_2026-10-17_r022.csv')
base = grupos_sensibilidad.loc[grupos_sensibilidad['escenario'].eq('base_4_variables') & grupos_sensibilidad['k'].eq(2)]
comparaciones_destino = []
for nombre in sorted(grupos_sensibilidad.loc[grupos_sensibilidad['escenario'].str.startswith('sin_destino_'), 'escenario'].unique()):
    escenario = grupos_sensibilidad.loc[grupos_sensibilidad['escenario'].eq(nombre) & grupos_sensibilidad['k'].eq(2)]
    comunes = base.merge(escenario, on='id_ruta', suffixes=('_base', '_sin_destino'))
    comparaciones_destino.append({'escenario': nombre, 'rutas_comunes': len(comunes), 'ari_vs_base': adjusted_rand_score(comunes['grupo_numerico_sin_orden_base'], comunes['grupo_numerico_sin_orden_sin_destino'])})
assert len(comparaciones_destino) == 3
assert len(tabla_sensibilidad.loc[tabla_sensibilidad['escenario'].eq('base_4_variables') & tabla_sensibilidad['k'].eq(2)]) == 1
display(pd.DataFrame(comparaciones_destino).round(3))
print('Ensayo exploratorio solamente: inventario sin congelar y sin etiquetas de dificultad.')

,escenario,rutas_comunes,ari_vs_base
0,sin_destino_d004,12,1.0
1,sin_destino_d009,12,1.0
2,sin_destino_d010,9,1.0


Ensayo exploratorio solamente: inventario sin congelar y sin etiquetas de dificultad.


## Comparación por destino exacto para la futura interfaz

Solo se comparan alternativas que llegan al **mismo destino final exacto** (`01_documentacion/destinos_exactos_interfaz.csv`), no todas las de un `id_destino` amplio: en Mindo, Cascada Corazón, Mirador y Yellow House son destinos distintos; en Pondoña, el circuito y la parte alta también. La función de recomendación permanece bloqueada hasta disponer de un modelo final validado y niveles interpretados. El ejemplo de Yellow House incluye rutas con tramos compartidos; no implica acceso actual ni seguridad.

In [14]:
import sys
if str(proyecto / '06_scripts') not in sys.path:
    sys.path.insert(0, str(proyecto / '06_scripts'))
from logica_recomendacion import alternativas_por_destino, cargar_destinos_exactos, recomendar, Preferencias, ModeloNoValidado
destinos_exactos = cargar_destinos_exactos()
# Mindo (D010) es solo una categoría amplia: mezclaría Cascada Corazón, Mirador y Yellow House.
try:
    alternativas_por_destino(dataset, 'D010', destinos_exactos)
except ValueError as error:
    print('Rechazado correctamente:', str(error)[:90], '...')
else:
    raise AssertionError('Un destino amplio con varios destinos exactos no puede compararse')
assert alternativas_por_destino(dataset, 'D010', destinos_exactos, 'Cascada Corazón (Mindo Hermit Trail)')['id_ruta'].tolist() == ['R015']
assert alternativas_por_destino(dataset, 'D010', destinos_exactos, 'Mirador (Mindo Hermit Trail)')['id_ruta'].tolist() == ['R016']
alternativas_mindo = alternativas_por_destino(dataset, 'D010', destinos_exactos, 'Yellow House')
assert alternativas_mindo['id_ruta'].tolist() == ['R017', 'R020', 'R021']
assert alternativas_mindo['destino_exacto'].nunique() == 1
try:
    recomendar(alternativas_mindo, Preferencias(nivel_deseado=2), None, version_modelo_final=None)
except ModeloNoValidado:
    print('Recomendación bloqueada correctamente: no existe modelo final validado.')
else:
    raise AssertionError('No debe haber recomendación sin modelo final')
display(alternativas_mindo[['id_ruta', 'destino_exacto', 'nombre_ruta', 'distancia_total_km', 'ascenso_total_m', 'pendiente_media_abs_pct']])

Rechazado correctamente: El destino amplio tiene varios destinos exactos; indique destino_exacto: Cascada Corazón ( ...
Recomendación bloqueada correctamente: no existe modelo final validado.


,id_ruta,destino_exacto,nombre_ruta,distancia_total_km,ascenso_total_m,pendiente_media_abs_pct
0,R017,Yellow House,Yellow House Trail,5.469,279.978,10.238
1,R020,Yellow House,Yellow House Trail vía Sendero 3,6.633,339.038,10.222
2,R021,Yellow House,Yellow House Trail vía Senderos 1 y 2,5.805,289.526,9.975


## Sensibilidad a distancia y ascenso

Como ambas variables están fuertemente correlacionadas, el ensayo exploratorio repite el agrupamiento sin una u otra. La ausencia de una solución válida con los filtros definidos no se rellena con grupos artificiales.

In [15]:
escenarios_variables = tabla_sensibilidad.loc[tabla_sensibilidad['escenario'].str.startswith('sin_variable_')]
assert 'sin_variable_ascenso' in set(escenarios_variables['escenario'])
assert 'sin_variable_distancia' in set(escenarios_variables['escenario'])
display(escenarios_variables[['escenario', 'n_rutas', 'k', 'silueta', 'tamano_minimo', 'semillas_validas']])
sin_ascenso = grupos_sensibilidad.loc[grupos_sensibilidad['escenario'].eq('sin_variable_ascenso') & grupos_sensibilidad['k'].eq(2)]
comunes_variables = base.merge(sin_ascenso, on='id_ruta', suffixes=('_base', '_sin_ascenso'))
ari_variables = adjusted_rand_score(comunes_variables['grupo_numerico_sin_orden_base'], comunes_variables['grupo_numerico_sin_orden_sin_ascenso'])
print(f'Con 14 rutas hay particiones sin distancia y sin ascenso; sin ascenso, ARI frente a base={ari_variables:.3f}. No se eligen variables finales.')

,escenario,n_rutas,k,silueta,tamano_minimo,semillas_validas
24,sin_variable_ascenso,14,2,0.3744,7,30
25,sin_variable_ascenso,14,3,0.3890,3,30
26,sin_variable_ascenso,14,4,0.4154,3,30
27,sin_variable_distancia,14,2,0.4738,2,30
28,sin_variable_distancia,14,3,0.4774,2,30
29,sin_variable_distancia,14,4,0.4400,2,26


Con 14 rutas hay particiones sin distancia y sin ascenso; sin ascenso, ARI frente a base=0.139. No se eligen variables finales.


## Control histórico del 21 de octubre de 2026

La auditoría de integridad comprueba archivos y cálculos, pero no reemplaza la verificación de acceso, identidad y seguridad en campo. En este corte todavía faltaban rutas para la meta de 20 y la clasificación no se había cerrado. El resultado K-means posterior se presenta en `05_validacion_metodologica.ipynb`; las recomendaciones de salida continúan bloqueadas por falta de verificación operativa.

In [16]:
auditoria_actual = pd.read_csv(proyecto / '07_resultados' / 'tablas' / 'auditoria_integridad_proyecto.csv')
puerta_actual = pd.read_csv(proyecto / '07_resultados' / 'tablas' / 'diagnostico_preparacion_modelo.csv').set_index('control')
assert len(auditoria_actual) > 0
auditoria_datos = auditoria_actual.loc[~auditoria_actual['ambito'].isin(['notebook_ejecucion', 'notebook_vigencia'])]
assert auditoria_datos['estado'].eq('OK').all()
elegibles_actuales = int(dataset['estado_modelado'].eq('elegible_actual').sum())
assert int(puerta_actual.loc['muestra_minima', 'valor']) == elegibles_actuales
faltantes_actuales = max(0, 20 - elegibles_actuales)
estado_puerta = puerta_actual.loc['muestra_minima', 'estado']
print(f'Corte de datos vigente: {len(dataset)} rutas procesadas, {elegibles_actuales} elegibles provisionales; faltan {faltantes_actuales} para la meta mínima de 20.')
print(f'Integridad de datos y código: {len(auditoria_datos)}/{len(auditoria_datos)} controles OK. Puerta de muestra: {estado_puerta}.')
print('La auditoría de ejecución de notebooks se repite después de guardar esta salida.')

Corte de datos vigente: 27 rutas procesadas, 20 elegibles provisionales; faltan 0 para la meta mínima de 20.
Integridad de datos y código: 257/257 controles OK. Puerta de muestra: OK.
La auditoría de ejecución de notebooks se repite después de guardar esta salida.


## Control de procedencia de R022 — 21 de octubre de 2026

El XML OSM exacto archivado el 20 de octubre se contrasta con el JSON normalizado y el GeoJSON procesado. El control verifica datos, no acredita apertura pública ni transitabilidad de los dos posibles vados marcados en OSM. R022 permanece provisional para el análisis, pero no se recomienda a usuarios.

In [17]:
from contrastar_r022_xml_snapshot import contrastar
fuente_r022 = contrastar()
assert fuente_r022['nodos'] == 128
assert fuente_r022['vados_osm'] == ['9996501327', '9996501328']
assert 'R022_xml_snapshot_geojson' in set(auditoria_actual['elemento'])
assert auditoria_actual.set_index('elemento').loc['R022_xml_snapshot_geojson', 'estado'] == 'OK'
print('R022: XML, snapshot y trazado concuerdan; acceso y vados sin validar para recomendación.')
print(f'Inventario: {len(dataset)} rutas; {elegibles_actuales} elegibles provisionales de 20; faltan {faltantes_actuales}.')

R022: XML, snapshot y trazado concuerdan; acceso y vados sin validar para recomendación.
Inventario: 27 rutas; 20 elegibles provisionales de 20; faltan 0.


## R023 Guagrapamba y corte vigente — 21 de octubre de 2026

Se contrasta el XML OSM archivado y fijado por SHA-256 con el sendero que termina en la cascada. La fuente municipal de 2018 menciona la caminata a la cascada, pero no verifica apertura ni condiciones actuales. R023 es elegible provisional solo para el análisis; no se recomienda al usuario. E34 y el ensayo de agrupamiento son exploratorios.

In [18]:
from procesar_r023_guagrapamba import construir as construir_r023
linea_r023, fuente_r023 = construir_r023()
assert len(fuente_r023['refs']) == 26 and fuente_r023['refs'][-1] == '8146768136'
assert 0 <= fuente_r023['distancia_inicio_parqueadero_m'] < 30
assert 'R023' in set(dataset['id_ruta'])
assert dataset.set_index('id_ruta').loc['R023', 'estado_modelado'] == 'elegible_actual'
assert len(dataset) >= 23 and int(dataset['estado_modelado'].eq('elegible_actual').sum()) >= 15
eda_historico_r023 = proyecto / '01_documentacion' / 'eda_preliminar_15_rutas_2026-10-21.md'
agrupamiento_r023 = pd.read_csv(proyecto / '07_resultados' / 'tablas' / 'evaluacion_clustering_exploratorio_sklearn_2026-10-21_r023.csv')
assert eda_historico_r023.exists()
assert agrupamiento_r023.loc[agrupamiento_r023['escenario'].eq('base_4_variables'), 'n_rutas'].eq(15).all()
assert agrupamiento_r023.loc[agrupamiento_r023['escenario'].eq('sin_r023_guagrapamba_pendiente'), 'n_rutas'].eq(14).all()
print(f'R023: {len(fuente_r023["refs"])} vértices; {fuente_r023["distancia_ida_km"]:.3f} km ida; extremo en cascada.')
print('Corte tras R023 (histórico): 15 elegibles provisionales; el corte vigente se registra en la celda siguiente.')

R023: 26 vértices; 1.669 km ida; extremo en cascada.
Corte tras R023 (histórico): 15 elegibles provisionales; el corte vigente se registra en la celda siguiente.


## R001 Rucu Pichincha rehabilitada y corte vigente — 21 de octubre de 2026

R001 estaba fuera del modelado porque su carácter no técnico estaba `por_verificar`. El 21 de octubre se documentó con la infografía del Ministerio de Turismo, las etiquetas SAC de OSM, SummitPost y prensa de 2026 (`01_documentacion/cierre_r001_rucu_pichincha_2026-10-21.md`). Es elegible provisional solo para el análisis: no se recomienda al usuario, tiene un tramo T3 y su acceso no está validado. E35 y el ensayo `r001` son exploratorios.

In [19]:
inventario_r001 = inventario.set_index('id_ruta').loc['R001']
assert inventario_r001['requiere_equipo_tecnico'] == 'no' and inventario_r001['comparable_trekking'] == 'si'
assert 'turismo.gob.ec' in inventario_r001['motivo_compatibilidad'] and 'sac_scale' in inventario_r001['motivo_compatibilidad']
assert dataset.set_index('id_ruta').loc['R001', 'estado_modelado'] == 'elegible_actual'
assert len(dataset) >= 23 and int(dataset['estado_modelado'].eq('elegible_actual').sum()) >= 16
assert (proyecto / '01_documentacion' / 'cierre_r001_rucu_pichincha_2026-10-21.md').exists()
assert (proyecto / '01_documentacion' / 'eda_preliminar_16_rutas_2026-10-21.md').exists()
agrupamiento_r001 = pd.read_csv(proyecto / '07_resultados' / 'tablas' / 'evaluacion_clustering_exploratorio_sklearn_2026-10-21_r001.csv')
assert agrupamiento_r001.loc[agrupamiento_r001['escenario'].eq('base_4_variables'), 'n_rutas'].eq(16).all()
print('Corte tras R001 (histórico): 23 procesadas, 16 elegibles provisionales; el corte vigente se registra en la celda siguiente.')


Corte tras R001 (histórico): 23 procesadas, 16 elegibles provisionales; el corte vigente se registra en la celda siguiente.


## R024 y R025 Cerro de la Luz y corte vigente — 21 de octubre de 2026

Dos senderos OSM peatonales (`path` y `footway`) llevan desde la vía «Entrada al Bosque Mágico» hasta el mismo nodo del Mirador Cerro de la Luz (Puéllaro): uno directo (0,5 km) y otro superior (1,3 km) que comparten solo el inicio y 51 m finales. El XML está archivado y fijado por SHA-256; los empalmes son nodos OSM compartidos. Son elegibles provisionales solo para el análisis: el trazado fue creado en 2025 por un solo editor, el acceso al Bosque Mágico no está confirmado y no se recomiendan al usuario. Se comparan únicamente entre sí por destino exacto.

In [20]:
from procesar_r024_r025_cerro_luz import construir as construir_cerro_luz
rutas_luz = construir_cerro_luz()
assert set(rutas_luz) == {'R024', 'R025'}
assert all(d['refs'][-1] == '11204680653' for _, d in rutas_luz.values())
assert set(inventario.set_index('id_ruta').loc[['R024', 'R025'], 'id_destino']) == {'D014'}
assert dataset.set_index('id_ruta').loc[['R024', 'R025'], 'estado_modelado'].eq('elegible_actual').all()
assert len(dataset) >= 25 and int(dataset['estado_modelado'].eq('elegible_actual').sum()) >= 18  # corte histórico de R024/R025
print(f"R024: {rutas_luz['R024'][1]['distancia_ida_km']:.3f} km ida; R025: {rutas_luz['R025'][1]['distancia_ida_km']:.3f} km ida; mismo mirador.")
print('Corte del 21 de octubre: 25 procesadas, 18 elegibles provisionales (histórico; el corte vigente está en las celdas de R026).')


R024: 0.505 km ida; R025: 1.275 km ida; mismo mirador.
Corte del 21 de octubre: 25 procesadas, 18 elegibles provisionales (histórico; el corte vigente está en las celdas de R026).


## R026 Tarabita – Cascada Guarumos (corte histórico) — 22 de octubre de 2026

Sendero OSM peatonal (`footway`, Ruta 1 → Ruta 3 → ramal final) que parte de la estación de desembarque de la tarabita (lado del santuario, la misma de R005 y R019) y llega al nodo «Cascada Guarumos» por nodos compartidos, sin conectores. El XML está archivado y fijado por SHA-256. Cascada Colibríes queda 44 m más allá y no se cuenta como ruta aparte; R026 comparte el 98 % de los vértices del trazado preliminar de R005, que sigue no elegible. Cruzar la tarabita (tarifa y horario de terceros por confirmar) es previo al sendero. Es elegible provisional solo para el análisis; no se recomienda al usuario.

In [21]:
from procesar_r026_guarumos import construir as construir_guarumos
linea_guarumos, datos_guarumos = construir_guarumos()
assert datos_guarumos['refs'][0] == '352972925' and datos_guarumos['refs'][-1] == '3047618576'
assert inventario.set_index('id_ruta').loc['R026', 'id_destino'] == 'D015'
assert dataset.set_index('id_ruta').loc['R026', 'estado_modelado'] == 'elegible_actual'
assert len(dataset) >= 26 and int(dataset['estado_modelado'].eq('elegible_actual').sum()) >= 19
print(f"R026: {datos_guarumos['distancia_ida_km']:.3f} km ida hasta Cascada Guarumos.")
print('Corte histórico R026: 26 procesadas y 19 elegibles; el corte vigente se revisa en R027.')

R026: 1.066 km ida hasta Cascada Guarumos.
Corte histórico R026: 26 procesadas y 19 elegibles; el corte vigente se revisa en R027.


## R027 Sendero del Viento, Moraspungo — corte de 20 rutas

La guía independiente describe un sendero peatonal adoquinado de unos 750 m y sin equipo especializado. La vía OSM original mide 728 m y tiene 32 nodos continuos desde la vía de acceso hasta el sector recreativo. El POI mirador está separado 1,7 m de la línea y no se usó un conector. La ruta es elegible provisional para analizar esfuerzo; no acredita horarios, estado actual ni accesibilidad en campo. Alcanzar 20 no habilita niveles ni recomendaciones.

In [22]:
from procesar_r027_sendero_viento import construir as construir_viento
coords_viento, datos_viento = construir_viento()
assert datos_viento['refs'][0] == '347634309' and datos_viento['refs'][-1] == '6797699199'
assert inventario.set_index('id_ruta').loc['R027', 'id_destino'] == 'D016'
assert dataset.set_index('id_ruta').loc['R027', 'estado_modelado'] == 'elegible_actual'
assert len(dataset) == 27 and int(dataset['estado_modelado'].eq('elegible_actual').sum()) == 20
print(f"R027: {datos_viento['distancia_ida_km']:.3f} km ida, {len(coords_viento)} nodos; mirador separado {datos_viento['separacion_mirador_m']:.1f} m.")
print('Corte vigente de inventario: 27 procesadas, 20 elegibles provisionales, 0 habilitadas para recomendación; la clasificación K-means vigente se valida en el notebook 05.')

R027: 0.728 km ida, 32 nodos; mirador separado 1.7 m.
Corte vigente de inventario: 27 procesadas, 20 elegibles provisionales, 0 habilitadas para recomendación; la clasificación K-means vigente se valida en el notebook 05.
